download dependancies

In [ ]:
!pip install torch --upgrade

In [ ]:
!pip install roboflow

In [ ]:
!pip install ultralytics

Download dataset

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="")   # Obtain your API key from Roboflow
project = rf.workspace("skyfusion-92vfy").project("prj2-bbnxz")
version = project.version(7)
dataset = version.download("yolov11")

upload modify_yolo.zip, it should contain `__init__.py `, `tasks.py`, the attention modules and `yolo11.yaml`.

In [ ]:
!unzip modify_yolo.zip

In [ ]:
!rm /usr/local/lib/python3.11/dist-packages/ultralytics/nn/modules/__init__.py
!mv /content/modify_yolo/__init__.py /usr/local/lib/python3.11/dist-packages/ultralytics/nn/modules/
!mv /content/modify_yolo/GAM_Attention.py /usr/local/lib/python3.11/dist-packages/ultralytics/nn/modules
!mv /content/modify_yolo/eca.py /usr/local/lib/python3.11/dist-packages/ultralytics/nn/modules
!rm /usr/local/lib/python3.11/dist-packages/ultralytics/nn/tasks.py
!mv /content/modify_yolo/tasks.py /usr/local/lib/python3.11/dist-packages/ultralytics/nn/
!rm /usr/local/lib/python3.11/dist-packages/ultralytics/cfg/models/11/yolo11.yaml
!mv /content/modify_yolo/yolo11.yaml /usr/local/lib/python3.11/dist-packages/ultralytics/cfg/models/11/

Run the models

Run model for 150 epochs

In [ ]:
from ultralytics import YOLO

model = YOLO("/usr/local/lib/python3.11/dist-packages/ultralytics/cfg/models/11/yolo11s.yaml")
results = model.train(data="/content/prj2-7/data.yaml", epochs=150, imgsz=640)

Run the below code in case the model needed to be stopped in between, else don't run it.

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/last.pt")
results = model.train(data="/content/prj2-7/data.yaml", save_dir="/runs/detect/train",epochs=150, imgsz=640,resume=True)

Run validation

In [ ]:
metrics = model.val(save_json=True)

In [ ]:
print(metrics.box.map50) #map50
print(metrics.box.map)#map90-95

Finally, download everything

In [ ]:
#@title Utility to zip and download a directory
#@markdown Use this method to zip and download a directory. For ex. a TB logs
#@markdown directory or a checkpoint(s) directory.

from google.colab import files
import os

dir_to_zip = './runs/detect/train/' #@param {type: "string"}
output_filename = 'yolo_11s_modified.zip' #@param {type: "string"}
delete_dir_after_download = "No"  #@param ['Yes', 'No']

os.system( "zip -r {} {}".format( output_filename , dir_to_zip ) )

if delete_dir_after_download == "Yes":
    os.system( "rm -r {}".format( dir_to_zip ) )

files.download(output_filename)